# Excel Data Analysis & Database Design

## Senior Data Engineer Analysis

This notebook performs comprehensive analysis of all uploaded Excel files to:

- Examine all sheets and columns
- Detect data types and patterns
- Identify primary and foreign keys
- Find relationships between tables
- Detect missing values and duplicates
- Create a complete data dictionary
- Design a PostgreSQL database schema


In [1]:
import pandas as pd
import openpyxl
from openpyxl import load_workbook
import os
from pathlib import Path
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Setup workspace path
workspace_path = r'c:\Users\shiva\OneDrive\Desktop\nifty_100project'
excel_files = [f for f in os.listdir(workspace_path) if f.endswith('.xlsx')]
print(f"Found {len(excel_files)} Excel files:")
for f in sorted(excel_files):
    print(f"  - {f}")

Found 7 Excel files:
  - analysis.xlsx
  - balancesheet.xlsx
  - cashflow.xlsx
  - companies.xlsx
  - documents.xlsx
  - profitandloss.xlsx
  - prosandcons.xlsx


## 1. Load and Read Excel Files

Loading all Excel files and analyzing their structure...


In [11]:
# Load all Excel files and store sheets
excel_data = {}

for excel_file in sorted(excel_files):
    file_path = os.path.join(workspace_path, excel_file)
    xl_file = pd.ExcelFile(file_path)
    sheet_names = xl_file.sheet_names
    
    print(f"\n📄 {excel_file}")
    print(f"   Sheets: {sheet_names}")
    
    excel_data[excel_file] = {}
    
    for sheet in sheet_names:
        df = pd.read_excel(file_path, sheet_name=sheet)
        excel_data[excel_file][sheet] = df
        print(f"   - {sheet}: {df.shape[0]} rows × {df.shape[1]} columns")

# Create a unified view of all tables
all_tables = {}
for excel_file, sheets in excel_data.items():
    for sheet_name, df in sheets.items():
        table_name = f"{excel_file.replace('.xlsx', '')}_{sheet_name}" if len(sheets) > 1 else excel_file.replace('.xlsx', '')
        all_tables[table_name] = df

print(f"\n✓ Total tables loaded: {len(all_tables)}")


📄 analysis.xlsx
   Sheets: ['Analysis']
   - Analysis: 21 rows × 6 columns

📄 balancesheet.xlsx
   Sheets: ['Balance Sheet']
   - Balance Sheet: 1313 rows × 13 columns

📄 cashflow.xlsx
   Sheets: ['Cash Flow']
   - Cash Flow: 1188 rows × 7 columns

📄 companies.xlsx
   Sheets: ['Companies']
   - Companies: 93 rows × 12 columns

📄 documents.xlsx
   Sheets: ['Documents']
   - Documents: 1586 rows × 4 columns

📄 profitandloss.xlsx
   Sheets: ['Profit & Loss']
   - Profit & Loss: 1277 rows × 15 columns

📄 prosandcons.xlsx
   Sheets: ['Pros & Cons']
   - Pros & Cons: 17 rows × 4 columns

✓ Total tables loaded: 7


## 2. Display Columns and Data Types

Analyzing all columns, their data types, and statistics...


In [3]:
# Analyze columns and data types for all tables
table_info = {}

for table_name, df in all_tables.items():
    print(f"\n{'='*80}")
    print(f"📊 Table: {table_name}")
    print(f"{'='*80}")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
    
    columns_info = []
    for col in df.columns:
        dtype = str(df[col].dtype)
        non_null = df[col].count()
        null_count = df[col].isna().sum()
        null_pct = (null_count / len(df)) * 100 if len(df) > 0 else 0
        
        # Try to identify column type
        if 'int' in dtype:
            col_type = 'INTEGER'
        elif 'float' in dtype:
            col_type = 'FLOAT'
        elif 'datetime' in dtype or 'date' in df[col].dtype.name.lower():
            col_type = 'DATE/DATETIME'
        elif dtype == 'object':
            # Check if it's likely a string with limited unique values
            unique_count = df[col].nunique()
            col_type = f'VARCHAR({df[col].astype(str).str.len().max()})' if unique_count > 10 else 'VARCHAR/ENUM'
        else:
            col_type = dtype.upper()
        
        columns_info.append({
            'Column': col,
            'Data Type': dtype,
            'SQL Type': col_type,
            'Non-Null': non_null,
            'Null': null_count,
            'Null %': f"{null_pct:.1f}%",
            'Unique': df[col].nunique()
        })
    
    columns_df = pd.DataFrame(columns_info)
    print(columns_df.to_string(index=False))
    table_info[table_name] = columns_info


📊 Table: analysis
Shape: 21 rows × 6 columns

                                                   Column Data Type     SQL Type  Non-Null  Null Null %  Unique
Bluestock Fintech — Nifty 100  |  Analysis  |  20 records    object   VARCHAR(2)        21     0   0.0%      21
                                               Unnamed: 1    object VARCHAR/ENUM        21     0   0.0%       6
                                               Unnamed: 2    object  VARCHAR(31)        21     0   0.0%      21
                                               Unnamed: 3    object  VARCHAR(24)        21     0   0.0%      19
                                               Unnamed: 4    object  VARCHAR(24)        21     0   0.0%      21
                                               Unnamed: 5    object  VARCHAR(21)        21     0   0.0%      21

📊 Table: balancesheet
Shape: 1313 rows × 13 columns

                                                           Column Data Type    SQL Type  Non-Null  Null Null %  Uni

## 3. Detect Missing Values and Duplicates

Analyzing data quality issues...


In [4]:
# Analyze missing values and duplicates
quality_report = {}

for table_name, df in all_tables.items():
    print(f"\n{'='*80}")
    print(f"🔍 Data Quality: {table_name}")
    print(f"{'='*80}")
    
    # Missing values analysis
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    
    print("\n📌 Missing Values:")
    if missing.sum() == 0:
        print("   ✓ No missing values detected")
    else:
        for col in df.columns:
            if missing[col] > 0:
                print(f"   - {col}: {missing[col]} ({missing_pct[col]:.2f}%)")
    
    # Duplicate rows analysis
    print("\n🔄 Duplicate Rows:")
    duplicates = df.duplicated().sum()
    print(f"   Total duplicate rows: {duplicates}")
    
    if duplicates > 0:
        dup_df = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
        print(f"   Showing first few duplicate rows:")
        print(dup_df.head(10).to_string())
    
    # Duplicate columns analysis
    print("\n🔀 Column Duplicates (same values in multiple columns):")
    col_duplicates = []
    for i, col1 in enumerate(df.columns):
        for col2 in df.columns[i+1:]:
            if df[col1].equals(df[col2]):
                col_duplicates.append((col1, col2))
    
    if col_duplicates:
        for col1, col2 in col_duplicates:
            print(f"   - {col1} == {col2}")
    else:
        print("   ✓ No duplicate columns")
    
    quality_report[table_name] = {
        'missing_values': missing,
        'duplicate_rows': duplicates,
        'duplicate_columns': col_duplicates
    }


🔍 Data Quality: analysis

📌 Missing Values:
   ✓ No missing values detected

🔄 Duplicate Rows:
   Total duplicate rows: 0

🔀 Column Duplicates (same values in multiple columns):
   ✓ No duplicate columns

🔍 Data Quality: balancesheet

📌 Missing Values:
   ✓ No missing values detected

🔄 Duplicate Rows:
   Total duplicate rows: 0

🔀 Column Duplicates (same values in multiple columns):
   ✓ No duplicate columns

🔍 Data Quality: cashflow

📌 Missing Values:
   - Unnamed: 3: 2 (0.17%)
   - Unnamed: 4: 2 (0.17%)
   - Unnamed: 5: 2 (0.17%)
   - Unnamed: 6: 2 (0.17%)

🔄 Duplicate Rows:
   Total duplicate rows: 0

🔀 Column Duplicates (same values in multiple columns):
   ✓ No duplicate columns

🔍 Data Quality: companies

📌 Missing Values:
   - Unnamed: 1: 1 (1.08%)
   - Unnamed: 5: 1 (1.08%)
   - Unnamed: 6: 1 (1.08%)
   - Unnamed: 7: 1 (1.08%)
   - Unnamed: 8: 1 (1.08%)
   - Unnamed: 9: 1 (1.08%)
   - Unnamed: 10: 1 (1.08%)
   - Unnamed: 11: 2 (2.15%)

🔄 Duplicate Rows:
   Total duplicate row

## 4. Identify Primary and Foreign Keys

Analyzing key candidates based on uniqueness and patterns...


In [5]:
# Identify candidate primary keys
key_analysis = {}

print("="*80)
print("🔑 PRIMARY KEY CANDIDATES")
print("="*80)

for table_name, df in all_tables.items():
    print(f"\n📋 {table_name}:")
    
    pk_candidates = []
    
    for col in df.columns:
        unique_count = df[col].nunique()
        total_rows = len(df)
        is_unique = unique_count == total_rows
        null_count = df[col].isna().sum()
        
        # Strong candidate if: unique, no nulls, and looks like ID
        is_id_like = any(keyword in col.lower() for keyword in ['id', 'code', 'key', 'number', 'pk'])
        
        if is_unique and null_count == 0:
            confidence = "HIGH"
            if is_id_like:
                pk_candidates.append((col, confidence))
                print(f"  ✓ STRONG: {col} (unique, no nulls, ID-like)")
            else:
                print(f"  ◐ POSSIBLE: {col} (unique, no nulls)")
    
    # If no obvious PK, suggest composite candidates
    if not pk_candidates:
        # Try combinations of 2-3 columns
        for col1 in df.columns[:min(5, len(df.columns))]:
            for col2 in df.columns[df.columns.get_loc(col1)+1:min(df.columns.get_loc(col1)+3, len(df.columns))]:
                composite_unique = df[[col1, col2]].drop_duplicates().shape[0]
                if composite_unique == len(df):
                    pk_candidates.append((f"{col1}+{col2}", "COMPOSITE"))
                    print(f"  ⚙ COMPOSITE: {col1} + {col2}")
                    break
    
    key_analysis[table_name] = {'pk_candidates': pk_candidates}

# Identify foreign key candidates
print(f"\n\n{'='*80}")
print("🔗 FOREIGN KEY CANDIDATES (across tables)")
print("="*80)

for table1_name in all_tables.keys():
    df1 = all_tables[table1_name]
    fk_candidates = []
    
    for col1 in df1.columns:
        # Look for similar columns in other tables
        for table2_name in all_tables.keys():
            if table1_name == table2_name:
                continue
            
            df2 = all_tables[table2_name]
            
            for col2 in df2.columns:
                # Check if column names are similar
                col1_normalized = col1.lower().replace('_', '')
                col2_normalized = col2.lower().replace('_', '')
                
                # Check for ID patterns
                is_id_pattern = any(x in col1_normalized for x in ['id', 'code']) and \
                                any(x in col2_normalized for x in ['id', 'code'])
                
                if is_id_pattern or col1_normalized == col2_normalized:
                    # Check if values in col1 exist in col2
                    col1_values = set(df1[col1].dropna().astype(str))
                    col2_values = set(df2[col2].dropna().astype(str))
                    
                    if len(col1_values) > 0 and col1_values.issubset(col2_values):
                        confidence = "HIGH" if len(col1_values) == len(col1_values & col2_values) else "MEDIUM"
                        fk_candidates.append({
                            'source_table': table1_name,
                            'source_column': col1,
                            'target_table': table2_name,
                            'target_column': col2,
                            'confidence': confidence
                        })
    
    if fk_candidates:
        print(f"\n  {table1_name}:")
        for fk in fk_candidates:
            print(f"    → {fk['source_column']} → {fk['target_table']}.{fk['target_column']} ({fk['confidence']})")
        key_analysis[table1_name]['fk_candidates'] = fk_candidates

🔑 PRIMARY KEY CANDIDATES

📋 analysis:
  ◐ POSSIBLE: Bluestock Fintech — Nifty 100  |  Analysis  |  20 records (unique, no nulls)
  ◐ POSSIBLE: Unnamed: 2 (unique, no nulls)
  ◐ POSSIBLE: Unnamed: 4 (unique, no nulls)
  ◐ POSSIBLE: Unnamed: 5 (unique, no nulls)
  ⚙ COMPOSITE: Bluestock Fintech — Nifty 100  |  Analysis  |  20 records + Unnamed: 1
  ⚙ COMPOSITE: Unnamed: 1 + Unnamed: 2
  ⚙ COMPOSITE: Unnamed: 2 + Unnamed: 3
  ⚙ COMPOSITE: Unnamed: 3 + Unnamed: 4
  ⚙ COMPOSITE: Unnamed: 4 + Unnamed: 5

📋 balancesheet:
  ◐ POSSIBLE: Bluestock Fintech — Nifty 100  |  Balance Sheet  |  1,312 records (unique, no nulls)
  ⚙ COMPOSITE: Bluestock Fintech — Nifty 100  |  Balance Sheet  |  1,312 records + Unnamed: 1

📋 cashflow:
  ◐ POSSIBLE: Bluestock Fintech — Nifty 100  |  Cash Flow  |  1,187 records (unique, no nulls)
  ⚙ COMPOSITE: Bluestock Fintech — Nifty 100  |  Cash Flow  |  1,187 records + Unnamed: 1

📋 companies:
  ◐ POSSIBLE: Mkt Fintech — Nifty 100  |  Companies  |  92 records (unique,

## 5. Analyze Relationships Between Files/Tables

Detecting common identifiers and relationship patterns...


In [6]:
# Analyze relationships between tables
relationships = []

print("="*80)
print("🔗 TABLE RELATIONSHIPS")
print("="*80)

table_names = list(all_tables.keys())

for i, table1 in enumerate(table_names):
    df1 = all_tables[table1]
    cols1 = set(df1.columns)
    
    for table2 in table_names[i+1:]:
        df2 = all_tables[table2]
        cols2 = set(df2.columns)
        
        # Find common columns
        common_cols = cols1 & cols2
        
        if common_cols:
            print(f"\n🔄 {table1} ↔ {table2}")
            print(f"   Common columns: {', '.join(common_cols)}")
            
            for col in common_cols:
                # Check cardinality
                vals1 = set(df1[col].dropna().astype(str))
                vals2 = set(df2[col].dropna().astype(str))
                common_vals = vals1 & vals2
                
                overlap_pct = (len(common_vals) / min(len(vals1), len(vals2)) * 100) if min(len(vals1), len(vals2)) > 0 else 0
                
                # Determine relationship type
                if len(vals1) == len(vals2) == len(common_vals):
                    rel_type = "1:1 (One-to-One)"
                elif len(vals1) < len(vals2) and len(common_vals) == len(vals1):
                    rel_type = "1:N (One-to-Many)"
                elif len(vals2) < len(vals1) and len(common_vals) == len(vals2):
                    rel_type = "N:1 (Many-to-One)"
                elif len(common_vals) > 0:
                    rel_type = "N:M (Many-to-Many)"
                else:
                    rel_type = "No relationship"
                
                print(f"   ├─ {col}")
                print(f"      {table1}: {len(vals1)} unique | {table2}: {len(vals2)} unique | Common: {len(common_vals)} ({overlap_pct:.1f}%)")
                print(f"      Type: {rel_type}")
                
                if overlap_pct > 50:  # Significant overlap
                    relationships.append({
                        'table1': table1,
                        'table2': table2,
                        'key': col,
                        'type': rel_type,
                        'strength': overlap_pct
                    })

if not relationships:
    print("\n⚠️  No direct relationships found based on common columns")

🔗 TABLE RELATIONSHIPS

🔄 analysis ↔ balancesheet
   Common columns: Unnamed: 4, Unnamed: 5, Unnamed: 2, Unnamed: 3, Unnamed: 1
   ├─ Unnamed: 4
      analysis: 21 unique | balancesheet: 1194 unique | Common: 0 (0.0%)
      Type: No relationship
   ├─ Unnamed: 5
      analysis: 21 unique | balancesheet: 1032 unique | Common: 0 (0.0%)
      Type: No relationship
   ├─ Unnamed: 2
      analysis: 21 unique | balancesheet: 55 unique | Common: 0 (0.0%)
      Type: No relationship
   ├─ Unnamed: 3
      analysis: 19 unique | balancesheet: 432 unique | Common: 0 (0.0%)
      Type: No relationship
   ├─ Unnamed: 1
      analysis: 6 unique | balancesheet: 99 unique | Common: 6 (100.0%)
      Type: 1:N (One-to-Many)

🔄 analysis ↔ cashflow
   Common columns: Unnamed: 4, Unnamed: 5, Unnamed: 2, Unnamed: 3, Unnamed: 1
   ├─ Unnamed: 4
      analysis: 21 unique | cashflow: 1063 unique | Common: 0 (0.0%)
      Type: No relationship
   ├─ Unnamed: 5
      analysis: 21 unique | cashflow: 1078 unique | C

## 6. Create Data Dictionary

Complete documentation of all tables and columns...


In [7]:
# Create comprehensive data dictionary
data_dictionary = []

print("="*100)
print("📖 COMPLETE DATA DICTIONARY")
print("="*100)

for table_name, df in all_tables.items():
    print(f"\n{'='*100}")
    print(f"TABLE: {table_name}")
    print(f"{'='*100}")
    print(f"Description: Data from {table_name} containing business information")
    print(f"Row Count: {len(df):,}")
    print(f"Column Count: {len(df.columns)}\n")
    
    for idx, col in enumerate(df.columns, 1):
        dtype = df[col].dtype
        non_null_count = df[col].count()
        null_count = df[col].isna().sum()
        unique_count = df[col].nunique()
        
        # Determine SQL type
        if 'int' in str(dtype):
            sql_type = 'BIGINT'
        elif 'float' in str(dtype):
            sql_type = 'NUMERIC(15,2)'
        elif 'datetime' in str(dtype):
            sql_type = 'TIMESTAMP'
        elif 'date' in str(dtype).lower():
            sql_type = 'DATE'
        else:
            max_len = df[col].astype(str).str.len().max()
            sql_type = f'VARCHAR({int(max_len * 1.2)})'
        
        # Determine constraints
        constraints = []
        if null_count == 0:
            constraints.append('NOT NULL')
        
        # Check if potential key
        if unique_count == len(df):
            constraints.append('UNIQUE')
        
        # Sample values
        sample_vals = df[col].dropna().unique()[:3]
        sample_str = ', '.join([str(v)[:20] for v in sample_vals])
        
        print(f"{idx:2d}. {col}")
        print(f"    Type (Python): {dtype}")
        print(f"    Type (SQL): {sql_type}")
        print(f"    Constraints: {', '.join(constraints) if constraints else 'None'}")
        print(f"    Non-Null: {non_null_count} ({(non_null_count/len(df)*100):.1f}%)")
        print(f"    Unique Values: {unique_count}")
        print(f"    Sample Values: {sample_str}")
        print()
        
        data_dictionary.append({
            'Table': table_name,
            'Column': col,
            'Data Type': dtype,
            'SQL Type': sql_type,
            'Constraints': ', '.join(constraints) if constraints else 'None',
            'Non-Null %': f"{(non_null_count/len(df)*100):.1f}%",
            'Unique Count': unique_count
        })

# Export data dictionary to DataFrame
dd_df = pd.DataFrame(data_dictionary)
print(f"\n{'='*100}")
print("DATA DICTIONARY SUMMARY")
print(f"{'='*100}\n")
print(dd_df.to_string(index=False))

📖 COMPLETE DATA DICTIONARY

TABLE: analysis
Description: Data from analysis containing business information
Row Count: 21
Column Count: 6

 1. Bluestock Fintech — Nifty 100  |  Analysis  |  20 records
    Type (Python): object
    Type (SQL): VARCHAR(2)
    Constraints: NOT NULL, UNIQUE
    Non-Null: 21 (100.0%)
    Unique Values: 21
    Sample Values: id, 1, 10

 2. Unnamed: 1
    Type (Python): object
    Type (SQL): VARCHAR(12)
    Constraints: NOT NULL
    Non-Null: 21 (100.0%)
    Unique Values: 6
    Sample Values: company_id, HDFCBANK, SBILIFE

 3. Unnamed: 2
    Type (Python): object
    Type (SQL): VARCHAR(37)
    Constraints: NOT NULL, UNIQUE
    Non-Null: 21 (100.0%)
    Unique Values: 21
    Sample Values: compounded_sales_gro, 10 Years: 21%, 5 Years:       24%

 4. Unnamed: 3
    Type (Python): object
    Type (SQL): VARCHAR(28)
    Constraints: NOT NULL
    Non-Null: 21 (100.0%)
    Unique Values: 19
    Sample Values: compounded_profit_gr, 10 Years: 22%, 5 Years:        

## 7. Design Database Schema

Defining table structures with appropriate constraints and indexes...


In [8]:
# Design database schema
schema_design = {}

print("="*100)
print("🗄️  DATABASE SCHEMA DESIGN")
print("="*100)

for table_name, df in all_tables.items():
    print(f"\n{'─'*100}")
    print(f"TABLE: {table_name}")
    print(f"{'─'*100}")
    print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
    print(f"Estimated Size: ~{(len(df) * len(df.columns) * 50 / 1024 / 1024):.2f} MB (rough estimate)\n")
    
    schema_design[table_name] = {
        'columns': [],
        'primary_key': None,
        'indexes': [],
        'constraints': []
    }
    
    # Identify primary key
    pk = None
    for col in df.columns:
        if df[col].nunique() == len(df) and df[col].isna().sum() == 0:
            if any(keyword in col.lower() for keyword in ['id', 'code', 'key', 'number']):
                pk = col
                break
    
    print(f"Primary Key: {pk if pk else 'RECOMMEND: Add surrogate ID'}")
    schema_design[table_name]['primary_key'] = pk
    
    # Column definitions
    print(f"\nColumns:")
    for col in df.columns:
        dtype = df[col].dtype
        null_count = df[col].isna().sum()
        unique_count = df[col].nunique()
        
        # Determine SQL type
        if 'int' in str(dtype):
            sql_type = 'BIGINT'
        elif 'float' in str(dtype):
            sql_type = 'NUMERIC(15,2)'
        elif 'datetime' in str(dtype):
            sql_type = 'TIMESTAMP'
        elif 'date' in str(dtype).lower():
            sql_type = 'DATE'
        else:
            max_len = df[col].astype(str).str.len().max()
            sql_type = f'VARCHAR({int(max_len * 1.2)})'
        
        # Constraints
        col_constraints = []
        if null_count == 0:
            col_constraints.append('NOT NULL')
        if col == pk:
            col_constraints.append('PRIMARY KEY')
        if unique_count == len(df):
            col_constraints.append('UNIQUE')
        
        constraints_str = ' '.join(col_constraints)
        
        print(f"  • {col}")
        print(f"    └─ {sql_type} {constraints_str}")
        
        schema_design[table_name]['columns'].append({
            'name': col,
            'sql_type': sql_type,
            'constraints': col_constraints
        })
    
    # Recommended indexes
    print(f"\nRecommended Indexes:")
    indexes = []
    
    # Index on PK
    if pk:
        print(f"  • PRIMARY KEY ({pk})")
        indexes.append(f"PRIMARY KEY ({pk})")
    
    # Indexes on frequently searched columns
    for col in df.columns:
        if col != pk and any(keyword in col.lower() for keyword in ['id', 'code', 'name', 'date']):
            print(f"  • INDEX: {col}")
            indexes.append(f"INDEX idx_{table_name}_{col} ({col})")
    
    schema_design[table_name]['indexes'] = indexes

🗄️  DATABASE SCHEMA DESIGN

────────────────────────────────────────────────────────────────────────────────────────────────────
TABLE: analysis
────────────────────────────────────────────────────────────────────────────────────────────────────
Rows: 21 | Columns: 6
Estimated Size: ~0.01 MB (rough estimate)

Primary Key: RECOMMEND: Add surrogate ID

Columns:
  • Bluestock Fintech — Nifty 100  |  Analysis  |  20 records
    └─ VARCHAR(2) NOT NULL UNIQUE
  • Unnamed: 1
    └─ VARCHAR(12) NOT NULL
  • Unnamed: 2
    └─ VARCHAR(37) NOT NULL UNIQUE
  • Unnamed: 3
    └─ VARCHAR(28) NOT NULL
  • Unnamed: 4
    └─ VARCHAR(28) NOT NULL UNIQUE
  • Unnamed: 5
    └─ VARCHAR(25) NOT NULL UNIQUE

Recommended Indexes:
  • INDEX: Unnamed: 1
  • INDEX: Unnamed: 2
  • INDEX: Unnamed: 3
  • INDEX: Unnamed: 4
  • INDEX: Unnamed: 5

────────────────────────────────────────────────────────────────────────────────────────────────────
TABLE: balancesheet
────────────────────────────────────────────────────

## 8. Generate Relationship Diagram

Creating an ER (Entity-Relationship) diagram...


In [9]:
# Generate ER Diagram using Mermaid notation
print("="*100)
print("📊 ENTITY-RELATIONSHIP DIAGRAM")
print("="*100 + "\n")

# Create Mermaid ER diagram
mermaid_diagram = "erDiagram\n"

for table_name, df in all_tables.items():
    # Identify primary key
    pk = None
    for col in df.columns:
        if df[col].nunique() == len(df) and df[col].isna().sum() == 0:
            if any(keyword in col.lower() for keyword in ['id', 'code', 'key', 'number']):
                pk = col
                break
    
    # Add table entity
    if pk:
        mermaid_diagram += f'    {table_name.upper()} {{\n'
        mermaid_diagram += f'        {pk} PK\n'
        for col in df.columns:
            if col != pk:
                mermaid_diagram += f'        {col}\n'
        mermaid_diagram += '    }\n'

# Add relationships
if relationships:
    mermaid_diagram += '\n'
    for rel in relationships[:5]:  # Limit to 5 relationships for clarity
        rel_symbol = '||' if rel['type'] == '1:1' else '|o'
        mermaid_diagram += f"    {rel['table1'].upper()} ||--{rel_symbol} {rel['table2'].upper()} : {rel['key']}\n"

print("Mermaid ER Diagram:")
print(mermaid_diagram)

# Create text-based ASCII diagram
print("\n" + "="*100)
print("TEXT-BASED DIAGRAM")
print("="*100 + "\n")

print("TABLE STRUCTURE OVERVIEW:\n")
for i, (table_name, df) in enumerate(all_tables.items(), 1):
    print(f"{i}. {table_name.upper()}")
    print(f"   ├─ Rows: {len(df):,}")
    print(f"   ├─ Columns: {len(df.columns)}")
    
    # Identify pk
    pk = None
    for col in df.columns:
        if df[col].nunique() == len(df) and df[col].isna().sum() == 0:
            if any(keyword in col.lower() for keyword in ['id', 'code', 'key', 'number']):
                pk = col
                break
    
    if pk:
        print(f"   ├─ Primary Key: {pk} 🔑")
    else:
        print(f"   ├─ Primary Key: [RECOMMEND: Add surrogate ID] ⚠️")
    
    print(f"   └─ Columns: {', '.join(df.columns[:5])}")
    if len(df.columns) > 5:
        print(f"      ... and {len(df.columns) - 5} more")
    print()

# Connection matrix
if relationships:
    print("\nRELATIONSHIP MATRIX:")
    print("─" * 100)
    for rel in relationships[:10]:
        print(f"{rel['table1']:20s} ──[{rel['key']:15s}]──> {rel['table2']:20s} ({rel['type']})")
else:
    print("\nNo relationships detected. Tables appear to be independent.")

📊 ENTITY-RELATIONSHIP DIAGRAM

Mermaid ER Diagram:
erDiagram

    ANALYSIS ||--|o BALANCESHEET : Unnamed: 1
    ANALYSIS ||--|o CASHFLOW : Unnamed: 1
    ANALYSIS ||--|o DOCUMENTS : Unnamed: 1
    ANALYSIS ||--|o PROFITANDLOSS : Unnamed: 1
    ANALYSIS ||--|o PROSANDCONS : Unnamed: 1


TEXT-BASED DIAGRAM

TABLE STRUCTURE OVERVIEW:

1. ANALYSIS
   ├─ Rows: 21
   ├─ Columns: 6
   ├─ Primary Key: [RECOMMEND: Add surrogate ID] ⚠️
   └─ Columns: Bluestock Fintech — Nifty 100  |  Analysis  |  20 records, Unnamed: 1, Unnamed: 2, Unnamed: 3, Unnamed: 4
      ... and 1 more

2. BALANCESHEET
   ├─ Rows: 1,313
   ├─ Columns: 13
   ├─ Primary Key: [RECOMMEND: Add surrogate ID] ⚠️
   └─ Columns: Bluestock Fintech — Nifty 100  |  Balance Sheet  |  1,312 records, Unnamed: 1, Unnamed: 2, Unnamed: 3, Unnamed: 4
      ... and 8 more

3. CASHFLOW
   ├─ Rows: 1,188
   ├─ Columns: 7
   ├─ Primary Key: [RECOMMEND: Add surrogate ID] ⚠️
   └─ Columns: Bluestock Fintech — Nifty 100  |  Cash Flow  |  1,187 reco

## 9. Output PostgreSQL Schema

Complete SQL statements for PostgreSQL database creation...


In [10]:
# Generate PostgreSQL CREATE TABLE statements
postgresql_schema = []

print("="*100)
print("🐘 POSTGRESQL SCHEMA DEFINITION")
print("="*100)

# First, create schema
schema_sql = """
-- ============================================================================
-- POSTGRESQL DATABASE SCHEMA
-- Generated from Excel data analysis
-- ============================================================================

-- Drop existing schema if exists (OPTIONAL)
-- DROP SCHEMA IF EXISTS nifty_100 CASCADE;

-- Create schema
CREATE SCHEMA IF NOT EXISTS nifty_100;

-- Set search path
SET search_path TO nifty_100, public;

-- ============================================================================
-- TABLE DEFINITIONS
-- ============================================================================
"""

print(schema_sql)

# Generate CREATE TABLE statements for each table
for table_name, df in all_tables.items():
    # Identify primary key
    pk = None
    for col in df.columns:
        if df[col].nunique() == len(df) and df[col].isna().sum() == 0:
            if any(keyword in col.lower() for keyword in ['id', 'code', 'key', 'number']):
                pk = col
                break
    
    # If no natural PK found, recommend adding one
    if not pk:
        pk = f"{table_name}_id"
    
    # Start CREATE TABLE statement
    create_table = f"\n-- Table: {table_name}\nCREATE TABLE nifty_100.{table_name} (\n"
    
    # If surrogate key needed
    if pk not in df.columns:
        create_table += f"    {pk} BIGSERIAL PRIMARY KEY,\n"
    
    # Add columns
    column_defs = []
    for col in df.columns:
        dtype = df[col].dtype
        null_count = df[col].isna().sum()
        
        # Determine SQL type with better size estimates
        if 'int' in str(dtype):
            sql_type = 'BIGINT'
        elif 'float' in str(dtype):
            sql_type = 'NUMERIC(15,2)'
        elif 'datetime' in str(dtype):
            sql_type = 'TIMESTAMP'
        elif 'date' in str(dtype).lower():
            sql_type = 'DATE'
        else:
            max_len = df[col].astype(str).str.len().max()
            estimated_len = max(50, int(max_len * 1.1))
            sql_type = f'VARCHAR({estimated_len})'
        
        # Build column definition
        col_def = f"    {col} {sql_type}"
        
        # Add constraints
        if null_count == 0:
            col_def += " NOT NULL"
        
        if col == pk and pk in df.columns:
            col_def += " PRIMARY KEY"
        elif df[col].nunique() == len(df) and col != pk:
            col_def += " UNIQUE"
        
        column_defs.append(col_def)
    
    # Combine column definitions
    create_table += ',\n'.join(column_defs)
    create_table += '\n);\n'
    
    # Add comments
    create_table += f"COMMENT ON TABLE nifty_100.{table_name} IS 'Contains {len(df):,} records with {len(df.columns)} columns';\n"
    
    # Add column comments
    for col in df.columns:
        unique_count = df[col].nunique()
        create_table += f"COMMENT ON COLUMN nifty_100.{table_name}.{col} IS '{unique_count} unique values';\n"
    
    # Add indexes
    for col in df.columns:
        if any(keyword in col.lower() for keyword in ['id', 'code', 'date', 'name']):
            create_table += f"CREATE INDEX idx_{table_name}_{col} ON nifty_100.{table_name} ({col});\n"
    
    print(create_table)
    postgresql_schema.append(create_table)

# Add foreign keys if relationships found
if relationships:
    print(f"\n-- ============================================================================")
    print(f"-- FOREIGN KEY CONSTRAINTS")
    print(f"-- ============================================================================\n")
    
    for rel in relationships[:5]:  # Limit to 5 FK relationships
        fk_col = rel['key']
        print(f"-- ALTER TABLE {rel['table1']} ADD CONSTRAINT fk_{rel['table1']}_{fk_col}")
        print(f"--   FOREIGN KEY ({fk_col}) REFERENCES {rel['table2']}({fk_col});\n")

# Final summary
print("\n" + "="*100)
print("SCHEMA SUMMARY")
print("="*100)
print(f"Total Tables: {len(all_tables)}")
print(f"Total Columns: {sum(len(df.columns) for df in all_tables.values())}")
print(f"Total Records: {sum(len(df) for df in all_tables.values()):,}")
print(f"Relationships Detected: {len(relationships)}")
print("="*100)

🐘 POSTGRESQL SCHEMA DEFINITION

-- ============================================================================
-- POSTGRESQL DATABASE SCHEMA
-- Generated from Excel data analysis
-- ============================================================================

-- Drop existing schema if exists (OPTIONAL)
-- DROP SCHEMA IF EXISTS nifty_100 CASCADE;

-- Create schema
CREATE SCHEMA IF NOT EXISTS nifty_100;

-- Set search path
SET search_path TO nifty_100, public;

-- ============================================================================
-- TABLE DEFINITIONS
-- ============================================================================


-- Table: analysis
CREATE TABLE nifty_100.analysis (
    analysis_id BIGSERIAL PRIMARY KEY,
    Bluestock Fintech — Nifty 100  |  Analysis  |  20 records VARCHAR(50) NOT NULL UNIQUE,
    Unnamed: 1 VARCHAR(50) NOT NULL,
    Unnamed: 2 VARCHAR(50) NOT NULL UNIQUE,
    Unnamed: 3 VARCHAR(50) NOT NULL,
    Unnamed: 4 VARCHAR(50) NOT NULL UNIQUE,
    Un

In [12]:
# DETAILED COLUMN EXTRACTION FOR SCHEMA GENERATION

print("="*100)
print("EXACT COLUMN NAMES AND DATA TYPES - FOR POSTGRESQL SCHEMA")
print("="*100)

for table_name, df in all_tables.items():
    print(f"\n{'='*100}")
    print(f"TABLE: {table_name.upper()}")
    print(f"{'='*100}")
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
    
    print("COLUMNS:")
    print("-" * 100)
    
    for idx, col in enumerate(df.columns, 1):
        pandas_dtype = str(df[col].dtype)
        
        # Get data samples
        non_null_vals = df[col].dropna()
        sample_vals = non_null_vals.head(3).tolist() if len(non_null_vals) > 0 else []
        
        # Analyze data to infer PostgreSQL type
        if 'int' in pandas_dtype:
            pg_type = 'BIGINT'
            sample_str = ', '.join([str(v) for v in sample_vals[:2]])
        elif 'float' in pandas_dtype:
            pg_type = 'NUMERIC(15,2)'
            sample_str = ', '.join([str(v) for v in sample_vals[:2]])
        elif 'datetime' in pandas_dtype:
            pg_type = 'TIMESTAMP'
            sample_str = str(sample_vals[0]) if sample_vals else 'N/A'
        else:
            # String/object type - determine appropriate VARCHAR length
            if len(non_null_vals) > 0:
                max_len = non_null_vals.astype(str).str.len().max()
                # Add 20% buffer for safety
                varchar_len = max(50, int(max_len * 1.2))
                pg_type = f'VARCHAR({varchar_len})'
                sample_str = ', '.join([f"'{v}'" for v in sample_vals[:2]])
            else:
                pg_type = 'VARCHAR(255)'
                sample_str = 'NULL'
        
        null_count = df[col].isna().sum()
        unique_count = df[col].nunique()
        
        print(f"{idx:2d}. Column Name: {col}")
        print(f"    Pandas Type: {pandas_dtype:20s} | PostgreSQL Type: {pg_type}")
        print(f"    Nulls: {null_count} ({null_count/len(df)*100:.1f}%) | Unique: {unique_count}")
        print(f"    Samples: {sample_str}")
        print()

print("\n" + "="*100)
print("SUMMARY - FOR SCHEMA.SQL GENERATION")
print("="*100)

column_mapping = {}

for table_name, df in all_tables.items():
    print(f"\n{table_name}:")
    column_mapping[table_name] = []
    
    for col in df.columns:
        pandas_dtype = str(df[col].dtype)
        
        # Determine PostgreSQL type
        if 'int' in pandas_dtype:
            pg_type = 'BIGINT'
        elif 'float' in pandas_dtype:
            pg_type = 'NUMERIC(15,2)'
        elif 'datetime' in pandas_dtype:
            pg_type = 'TIMESTAMP'
        else:
            non_null_vals = df[col].dropna()
            if len(non_null_vals) > 0:
                max_len = non_null_vals.astype(str).str.len().max()
                varchar_len = max(50, int(max_len * 1.2))
                pg_type = f'VARCHAR({varchar_len})'
            else:
                pg_type = 'VARCHAR(255)'
        
        null_count = df[col].isna().sum()
        is_unique = df[col].nunique() == len(df)
        has_nulls = null_count > 0
        
        column_mapping[table_name].append({
            'name': col,
            'pg_type': pg_type,
            'has_nulls': has_nulls,
            'is_unique': is_unique,
            'null_count': null_count
        })
        
        print(f"  {col:<30s} {pg_type:<25s} NULL:{has_nulls} UNIQUE:{is_unique}")

# Save for later use
import json
print("\n\nColumn mapping saved for schema generation...")

EXACT COLUMN NAMES AND DATA TYPES - FOR POSTGRESQL SCHEMA

TABLE: ANALYSIS
Shape: 21 rows × 6 columns

COLUMNS:
----------------------------------------------------------------------------------------------------
 1. Column Name: Bluestock Fintech — Nifty 100  |  Analysis  |  20 records
    Pandas Type: object               | PostgreSQL Type: VARCHAR(50)
    Nulls: 0 (0.0%) | Unique: 21
    Samples: 'id', '1'

 2. Column Name: Unnamed: 1
    Pandas Type: object               | PostgreSQL Type: VARCHAR(50)
    Nulls: 0 (0.0%) | Unique: 6
    Samples: 'company_id', 'HDFCBANK'

 3. Column Name: Unnamed: 2
    Pandas Type: object               | PostgreSQL Type: VARCHAR(50)
    Nulls: 0 (0.0%) | Unique: 21
    Samples: 'compounded_sales_growth', '10 Years: 21%'

 4. Column Name: Unnamed: 3
    Pandas Type: object               | PostgreSQL Type: VARCHAR(50)
    Nulls: 0 (0.0%) | Unique: 19
    Samples: 'compounded_profit_growth', '10 Years: 22%'

 5. Column Name: Unnamed: 4
    Pandas Type

In [14]:
# Check actual table names and extract exact columns

print("Available table names:")
for key in sorted(all_tables.keys()):
    print(f"  {key}")

# Companies - note: headers are in first row
print("\n" + "="*80)
print("COMPANIES")
print("="*80)
df_companies = all_tables['companies']
print("Columns:", df_companies.columns.tolist()[:3], "...")  # First 3 unnamed cols
print("\nActual headers (row 0):", df_companies.iloc[0].tolist()[:12])
print("Shape:", df_companies.shape)
print("\nData row 1:")
print(df_companies.iloc[1].tolist()[:5])

print("\n" + "="*80)
print("BALANCE_SHEET")
print("="*80)
df_bs = all_tables['balancesheet']
print("Columns:", df_bs.columns.tolist())
print("Shape:", df_bs.shape)
print("\nFirst row data:", df_bs.iloc[0].to_dict())

print("\n" + "="*80)
print("PROFIT_AND_LOSS")
print("="*80)
df_pl = all_tables['profitandloss']
print("Columns:", df_pl.columns.tolist())
print("Shape:", df_pl.shape)
print("\nFirst row data:", df_pl.iloc[0].to_dict())

print("\n" + "="*80)
print("CASH_FLOW")
print("="*80)
df_cf = all_tables['cashflow']
print("Columns:", df_cf.columns.tolist())
print("Shape:", df_cf.shape)
print("\nFirst row data:", df_cf.iloc[0].to_dict())

print("\n" + "="*80)
print("ANALYSIS")
print("="*80)
df_analysis = all_tables['analysis']
print("Columns:", df_analysis.columns.tolist())
print("Shape:", df_analysis.shape)
print("\nFirst row data:", df_analysis.iloc[0].to_dict())

print("\n" + "="*80)
print("DOCUMENTS")
print("="*80)
df_docs = all_tables['documents']
print("Columns:", df_docs.columns.tolist())
print("Shape:", df_docs.shape)
print("\nFirst row data:", df_docs.iloc[0].to_dict())

print("\n" + "="*80)
print("PROSANDCONS")
print("="*80)
df_pac = all_tables['prosandcons']
print("Columns:", df_pac.columns.tolist())
print("Shape:", df_pac.shape)
print("\nFirst row data:", df_pac.iloc[0].to_dict())

Available table names:
  analysis
  balancesheet
  cashflow
  companies
  documents
  profitandloss
  prosandcons

COMPANIES
Columns: ['Mkt Fintech — Nifty 100  |  Companies  |  92 records', 'Unnamed: 1', 'Unnamed: 2'] ...

Actual headers (row 0): ['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']
Shape: (93, 12)

Data row 1:
['ABB', 'https://mkt.in/static/mkt-icons/nifty100/ABB.png', 'Abbott India Ltd', 'https://in.tradingview.com/chart/?symbol=NSE%3AABBOTINDIA', 'Abbott India Ltd is one of the leading multinational pharmaceutical companies in India and sells its products through independent distributors primarily within India.']

BALANCE_SHEET
Columns: ['Bluestock Fintech — Nifty 100  |  Balance Sheet  |  1,312 records', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', '

In [15]:
# EXTRACT JUST THE COLUMN NAMES FOR SCHEMA GENERATION

import os
workspace_path = r'c:\Users\shiva\OneDrive\Desktop\nifty_100project'

schema_info = {}

# Companies
df = pd.read_excel(os.path.join(workspace_path, 'companies.xlsx'), sheet_name='Companies')
actual_headers = df.iloc[0].tolist()
df_clean = df.iloc[1:].copy()
df_clean.columns = actual_headers
schema_info['companies'] = {
    'columns': actual_headers,
    'rows': len(df_clean),
    'dtypes': df_clean.dtypes.to_dict()
}
print("COMPANIES Columns:", actual_headers)

# Balance Sheet
df = pd.read_excel(os.path.join(workspace_path, 'balancesheet.xlsx'), sheet_name='Balance Sheet')
schema_info['balance_sheet'] = {
    'columns': df.columns.tolist(),
    'rows': len(df),
    'dtypes': df.dtypes.to_dict()
}
print("\nBALANCE_SHEET Columns:", df.columns.tolist())

# Profit and Loss
df = pd.read_excel(os.path.join(workspace_path, 'profitandloss.xlsx'), sheet_name='Profit & Loss')
schema_info['profit_and_loss'] = {
    'columns': df.columns.tolist(),
    'rows': len(df),
    'dtypes': df.dtypes.to_dict()
}
print("\nPROFIT_AND_LOSS Columns:", df.columns.tolist())

# Cash Flow
df = pd.read_excel(os.path.join(workspace_path, 'cashflow.xlsx'), sheet_name='Cash Flow')
schema_info['cash_flow'] = {
    'columns': df.columns.tolist(),
    'rows': len(df),
    'dtypes': df.dtypes.to_dict()
}
print("\nCASH_FLOW Columns:", df.columns.tolist())

# Analysis
df = pd.read_excel(os.path.join(workspace_path, 'analysis.xlsx'), sheet_name='Analysis')
schema_info['analysis'] = {
    'columns': df.columns.tolist(),
    'rows': len(df),
    'dtypes': df.dtypes.to_dict()
}
print("\nANALYSIS Columns:", df.columns.tolist())

# Documents
df = pd.read_excel(os.path.join(workspace_path, 'documents.xlsx'), sheet_name='Documents')
schema_info['documents'] = {
    'columns': df.columns.tolist(),
    'rows': len(df),
    'dtypes': df.dtypes.to_dict()
}
print("\nDOCUMENTS Columns:", df.columns.tolist())

# Pros and Cons
df = pd.read_excel(os.path.join(workspace_path, 'prosandcons.xlsx'), sheet_name='Pros & Cons')
schema_info['prosandcons'] = {
    'columns': df.columns.tolist(),
    'rows': len(df),
    'dtypes': df.dtypes.to_dict()
}
print("\nPROSANDCONS Columns:", df.columns.tolist())

print("\n✓ Schema info extracted successfully")

COMPANIES Columns: ['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']

BALANCE_SHEET Columns: ['Bluestock Fintech — Nifty 100  |  Balance Sheet  |  1,312 records', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']

PROFIT_AND_LOSS Columns: ['Bluestock Fintech — Nifty 100  |  Profit & Loss  |  1,276 records', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']

CASH_FLOW Columns: ['Bluestock Fintech — Nifty 100  |  Cash Flow  |  1,187 records', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6']

ANALYSIS Columns: ['Bluestock Fintech — Nifty 100  |  Analysis  |  20 re

In [17]:
# SIMPLIFIED: Extract columns directly from files

col_info = {}

# Companies - skip header row
df = pd.read_excel(os.path.join(workspace_path, 'companies.xlsx'), skiprows=1)
col_info['companies'] = df.columns.tolist()
print("COMPANIES:")
print(col_info['companies'])
print(f"Rows: {len(df)}\n")

# Balance Sheet - skip header row
df = pd.read_excel(os.path.join(workspace_path, 'balancesheet.xlsx'), skiprows=1)
col_info['balance_sheet'] = df.columns.tolist()
print("BALANCE_SHEET:")
print(col_info['balance_sheet'])
print(f"Rows: {len(df)}\n")

# Profit and Loss - skip header row
df = pd.read_excel(os.path.join(workspace_path, 'profitandloss.xlsx'), skiprows=1)
col_info['profit_and_loss'] = df.columns.tolist()
print("PROFIT_AND_LOSS:")
print(col_info['profit_and_loss'])
print(f"Rows: {len(df)}\n")

# Cash Flow - skip header row
df = pd.read_excel(os.path.join(workspace_path, 'cashflow.xlsx'), skiprows=1)
col_info['cash_flow'] = df.columns.tolist()
print("CASH_FLOW:")
print(col_info['cash_flow'])
print(f"Rows: {len(df)}\n")

# Analysis - skip header row
df = pd.read_excel(os.path.join(workspace_path, 'analysis.xlsx'), skiprows=1)
col_info['analysis'] = df.columns.tolist()
print("ANALYSIS:")
print(col_info['analysis'])
print(f"Rows: {len(df)}\n")

# Documents - skip header row
df = pd.read_excel(os.path.join(workspace_path, 'documents.xlsx'), skiprows=1)
col_info['documents'] = df.columns.tolist()
print("DOCUMENTS:")
print(col_info['documents'])
print(f"Rows: {len(df)}\n")

# Pros and Cons - skip header row
df = pd.read_excel(os.path.join(workspace_path, 'prosandcons.xlsx'), skiprows=1)
col_info['prosandcons'] = df.columns.tolist()
print("PROSANDCONS:")
print(col_info['prosandcons'])
print(f"Rows: {len(df)}\n")

print("✓ All columns extracted")

COMPANIES:
['id', 'company_logo', 'company_name', 'chart_link', 'about_company', 'website', 'nse_profile', 'bse_profile', 'face_value', 'book_value', 'roce_percentage', 'roe_percentage']
Rows: 92

BALANCE_SHEET:
['id', 'company_id', 'year', 'equity_capital', 'reserves', 'borrowings', 'other_liabilities', 'total_liabilities', 'fixed_assets', 'cwip', 'investments', 'other_asset', 'total_assets']
Rows: 1312

PROFIT_AND_LOSS:
['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout']
Rows: 1276

CASH_FLOW:
['id', 'company_id', 'year', 'operating_activity', 'investing_activity', 'financing_activity', 'net_cash_flow']
Rows: 1187

ANALYSIS:
['id', 'company_id', 'compounded_sales_growth', 'compounded_profit_growth', 'stock_price_cagr', 'roe']
Rows: 20

DOCUMENTS:
['id', 'company_id', 'Year', 'Annual_Report']
Rows: 1585

PROSANDCONS:
['id', 'com

In [18]:
# INFER POSTGRESQL DATA TYPES FROM ACTUAL DATA

def get_pg_type(series, col_name):
    """Infer PostgreSQL data type from pandas Series"""
    dtype = series.dtype
    
    if 'int' in str(dtype):
        return 'BIGINT'
    elif 'float' in str(dtype):
        return 'NUMERIC(15,2)'
    elif 'datetime' in str(dtype):
        return 'TIMESTAMP'
    else:
        # String/object - infer VARCHAR length
        non_null = series.dropna()
        if len(non_null) > 0:
            max_len = non_null.astype(str).str.len().max()
            # Add buffer
            varchar_len = max(50, int(max_len * 1.2))
            return f'VARCHAR({varchar_len})'
        else:
            return 'VARCHAR(255)'

pg_schema = {}

# Companies
df = pd.read_excel(os.path.join(workspace_path, 'companies.xlsx'), skiprows=1)
pg_schema['companies'] = {col: get_pg_type(df[col], col) for col in df.columns}

# Balance Sheet
df = pd.read_excel(os.path.join(workspace_path, 'balancesheet.xlsx'), skiprows=1)
pg_schema['balance_sheet'] = {col: get_pg_type(df[col], col) for col in df.columns}

# Profit and Loss
df = pd.read_excel(os.path.join(workspace_path, 'profitandloss.xlsx'), skiprows=1)
pg_schema['profit_and_loss'] = {col: get_pg_type(df[col], col) for col in df.columns}

# Cash Flow
df = pd.read_excel(os.path.join(workspace_path, 'cashflow.xlsx'), skiprows=1)
pg_schema['cash_flow'] = {col: get_pg_type(df[col], col) for col in df.columns}

# Analysis
df = pd.read_excel(os.path.join(workspace_path, 'analysis.xlsx'), skiprows=1)
pg_schema['analysis'] = {col: get_pg_type(df[col], col) for col in df.columns}

# Documents
df = pd.read_excel(os.path.join(workspace_path, 'documents.xlsx'), skiprows=1)
pg_schema['documents'] = {col: get_pg_type(df[col], col) for col in df.columns}

# Prosandcons
df = pd.read_excel(os.path.join(workspace_path, 'prosandcons.xlsx'), skiprows=1)
pg_schema['prosandcons'] = {col: get_pg_type(df[col], col) for col in df.columns}

# Display inferred types
print("="*80)
print("INFERRED POSTGRESQL SCHEMA")
print("="*80)

for table_name in sorted(pg_schema.keys()):
    print(f"\n{table_name.upper()}:")
    for col_name, pg_type in pg_schema[table_name].items():
        print(f"  {col_name:<30s} {pg_type}")

print("\n✓ Schema inference complete")

INFERRED POSTGRESQL SCHEMA

ANALYSIS:
  id                             BIGINT
  company_id                     VARCHAR(50)
  compounded_sales_growth        VARCHAR(50)
  compounded_profit_growth       VARCHAR(50)
  stock_price_cagr               VARCHAR(50)
  roe                            VARCHAR(50)

BALANCE_SHEET:
  id                             BIGINT
  company_id                     VARCHAR(50)
  year                           VARCHAR(50)
  equity_capital                 NUMERIC(15,2)
  reserves                       BIGINT
  borrowings                     BIGINT
  other_liabilities              BIGINT
  total_liabilities              BIGINT
  fixed_assets                   BIGINT
  cwip                           BIGINT
  investments                    BIGINT
  other_asset                    BIGINT
  total_assets                   BIGINT

CASH_FLOW:
  id                             BIGINT
  company_id                     VARCHAR(50)
  year                           VARCHAR(50)
  